# Chess Laws RAG Pipeline

This notebook implements a Semantic Chunking / RAG pipeline for the FIDE Laws of Chess. We use `pymupdf4llm` to extract Markdown, Langchain's MarkdownHeaderTextSplitter to divide chunks along logical chapter/section bounds, inject "Breadcrumbs" into each chunk, then apply **Hybrid Search** (Dense + Sparse BM25) with **Cross-Encoder Reranking**.

In [1]:
%pip install -q pymupdf4llm langchain langchain-text-splitters langchain-community langchain-huggingface sentence-transformers chromadb langchain-chroma rank_bm25

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import json
import numpy as np
import pymupdf4llm
from typing import List
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder


In [3]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU Name: NVIDIA GeForce GTX 1650 with Max-Q Design


### Step 1: PDF to Markdown Extraction

In [4]:
pdf_path = "LawsOfChess.pdf"
pages_output_path = "chess.json"

# Extract with page_chunks=True to retain page numbers
if not os.path.exists(pages_output_path):
    print("Extracting markdown pages from PDF...")
    md_pages = pymupdf4llm.to_markdown(pdf_path, page_chunks=True)
    with open(pages_output_path, "w", encoding="utf-8") as f:
        json.dump(md_pages, f)
    print("Extraction complete and saved to", pages_output_path)
else:
    print("Loading existing markdown pages file...")
    with open(pages_output_path, "r", encoding="utf-8") as f:
        md_pages = json.load(f)

print(f"Loaded {len(md_pages)} pages.")


Loading existing markdown pages file...
Loaded 25 pages.


### Step 2: Semantic Chunking via Headers

We use `MarkdownHeaderTextSplitter` to separate out chunks using the `#` tags, and inject `page_number`, `version_number`, and `category` metadata.

In [5]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
    ("####", "Header 4"),
]

# Initial split by headers
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on, strip_headers=False
)

chunk_size = 1500
chunk_overlap = 200
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, chunk_overlap=chunk_overlap
)

all_header_splits = []
for page in md_pages:
    page_text = page['text']
    page_num = page['metadata'].get('page', 0)
    
    splits = markdown_splitter.split_text(page_text)
    
    # Inject Metadata
    for split in splits:
        split.metadata['page_number'] = page_num
        split.metadata['version_number'] = "2nd Edition"
        
        # Define Category dynamically from highest header
        if "Header 1" in split.metadata:
            split.metadata['category'] = split.metadata["Header 1"]
        elif "Header 2" in split.metadata:
            split.metadata['category'] = split.metadata["Header 2"]
        else:
            split.metadata['category'] = "General Rules"
            
    all_header_splits.extend(splits)

print(f"Discovered {len(all_header_splits)} header-bound semantic chunks.")

# Fallback splitter for very large chunks
final_splits = text_splitter.split_documents(all_header_splits)
print(f"Final number of chunks after character splitting: {len(final_splits)}")


Discovered 52 header-bound semantic chunks.
Final number of chunks after character splitting: 67


### Step 3: The Context Window Trick (Breadcrumbs)

We prepend the header context directly into the text so that the embedding model learns the structural hierarchy for every chunk.

In [6]:
def prepend_breadcrumbs(docs: List[Document]) -> List[Document]:
    prepended_docs = []
    for doc in docs:
        # Extract headers from metadata dict
        headers = []
        for h_level in ["Header 1", "Header 2", "Header 3", "Header 4"]:
            if h_level in doc.metadata:
                headers.append(doc.metadata[h_level])
        
        # Construct breadcrumb string
        if headers:
            breadcrumb_str = " > ".join(headers)
            # Prepend breadcrumbs to chunk page_content
            new_content = f"Breadcrumb Context: {breadcrumb_str}\n\n{doc.page_content}"
        else:
            new_content = doc.page_content
            
        # Create a new document with updated content
        prepended_docs.append(Document(page_content=new_content, metadata=doc.metadata))
        
    return prepended_docs

processed_splits = prepend_breadcrumbs(final_splits)

# Look at the first processed split
print("Sample chunk: \n")
print(processed_splits[1].page_content if len(processed_splits) > 1 else processed_splits[0].page_content)


Sample chunk: 

Breadcrumb Context: **FIDE LAWS of CHESS**

## **FIDE LAWS of CHESS**


### Step 4: Embedding and Dense Vector Store (ChromaDB)

We use a fast, open-source embedding model from HuggingFace and load it into a Chroma Vector Database.

In [7]:
print("Initializing embeddings model...")
embeddings_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embeddings_model_name, model_kwargs={'device': device})

print("Building Dense VectorStore with ChromaDB...")
persist_directory = "./chroma_chess_db"

vectorstore = Chroma.from_documents(
    documents=processed_splits,
    embedding=embeddings,
    persist_directory=persist_directory
)
print(f"Chroma (Dense) vector store saved to {persist_directory} directory.")


Initializing embeddings model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building Dense VectorStore with ChromaDB...
Chroma (Dense) vector store saved to ./chroma_chess_db directory.


### Step 5: Hybrid Search (Dense + Sparse BM25)

We combine **Dense Retrieval** (ChromaDB vector similarity) with **Sparse Retrieval** (BM25 keyword matching) to catch both semantic and exact-keyword matches.

In [8]:
# Build BM25 Index from the processed chunks
corpus = [doc.page_content for doc in processed_splits]
tokenized_corpus = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)
print(f"BM25 sparse index built over {len(corpus)} chunks.")

def hybrid_search(query: str, k_dense: int = 5, k_sparse: int = 5) -> List[Document]:
    """Combine Dense (Chroma) + Sparse (BM25) retrieval."""
    # Dense retrieval from ChromaDB
    dense_results = vectorstore.similarity_search(query, k=k_dense)
    
    # Sparse retrieval from BM25
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    top_sparse_indices = np.argsort(bm25_scores)[::-1][:k_sparse]
    sparse_results = [processed_splits[i] for i in top_sparse_indices]
    
    # Merge and deduplicate (by page_content)
    seen = set()
    merged = []
    for doc in dense_results + sparse_results:
        content_hash = hash(doc.page_content)
        if content_hash not in seen:
            seen.add(content_hash)
            merged.append(doc)
    
    return merged

# Quick test
test_results = hybrid_search("smoke grenades line of sight")
print(f"Hybrid search returned {len(test_results)} unique documents.")


BM25 sparse index built over 67 chunks.
Hybrid search returned 6 unique documents.


### Step 6: Cross-Encoder Reranking

We use a HuggingFace Cross-Encoder (`ms-marco-MiniLM-L-6-v2`) to re-score and re-order the hybrid results based on how well each chunk **actually answers** the query.

In [9]:
# Load the Cross-Encoder reranker model
print("Loading Cross-Encoder reranker...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=device)
print("Reranker loaded.")

def hybrid_search_with_rerank(query: str, top_n: int = 3) -> List[Document]:
    """Full pipeline: Hybrid Search -> Cross-Encoder Reranking."""
    # Step 1: Get candidates from hybrid search (dense + sparse)
    candidates = hybrid_search(query, k_dense=5, k_sparse=5)
    
    # Step 2: Score each candidate with the Cross-Encoder
    pairs = [[query, doc.page_content] for doc in candidates]
    scores = cross_encoder.predict(pairs)
    
    # Step 3: Sort by relevance score (highest first)
    scored_docs = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
    
    # Return top_n reranked results
    return [(score, doc) for score, doc in scored_docs[:top_n]]


Loading Cross-Encoder reranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reranker loaded.


### Step 7: Testing the Full Hybrid + Reranked Pipeline

In [10]:
question = "How does a pawn move?"
print(f"Query: {question}\n")

reranked_results = hybrid_search_with_rerank(question, top_n=3)

for i, (score, doc) in enumerate(reranked_results):
    print(f"--- Reranked Result {i+1} (score: {score:.4f}) ---")
    print(f"Metadata: {doc.metadata}")
    print(doc.page_content[:400] + "...")
    print("\n")


Query: How does a pawn move?

--- Reranked Result 1 (score: 5.1791) ---
Metadata: {'version_number': '2nd Edition', 'page_number': 0, 'category': 'General Rules'}
- 3.5 When making these moves the bishop, rook or queen may not move over any intervening pieces.  
- 3.6 The knight may move to one of the squares nearest to that on which it stands but not on the same rank, file or diagonal.  
**==> picture [150 x 150] intentionally omitted <==**  
- 3.7 a. The pawn may move forward to the unoccupied square immediately in front of it on the same file, or  
- b. ...


--- Reranked Result 2 (score: 1.6973) ---
Metadata: {'page_number': 0, 'version_number': '2nd Edition', 'category': 'General Rules'}
Examples:  
1. There are two knights, on the squares g1 and e1, and one of them moves to the square f3: either Ngf3 or Nef3, as the case may be.  
2. There are two knights, on the squares g5 and g1, and one of them moves to the square f3: either N5f3 or N1f3, as the case may be.  
3. There are two

### Step 8: LLM-Powered Answer Generation

We pass the top reranked chunks to an open-source LLM with a strict system prompt. The model is instructed to answer **only** from the provided context.

We use `huggingface_hub`'s `InferenceClient` to call open-source models (e.g. Mistral, Llama) hosted for free on HuggingFace's serverless inference API. The models themselves are fully open-source.

> **Local Alternative:** If you prefer fully offline inference, install [Ollama](https://ollama.com) and swap the client for `ollama.chat()`.

In [14]:
from transformers import pipeline, GenerationConfig
import torch

# Fully local, open-source LLM — no API key needed
print('Loading local LLM (TinyLlama-1.1B-Chat)...')
llm_pipeline = pipeline(
    'text-generation',
    model='TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    device=0 if device == 'cuda' else -1
)
print('Local LLM ready!')

SYSTEM_PROMPT = (
    "You are a Grandmaster Strategist. Using ONLY the provided ruleset excerpts, "
    "answer the user's question. If the answer isn't in the context, say you don't know."
)

def generate_answer(query: str, top_n: int = 3) -> str:
    """Full RAG pipeline: Hybrid Search -> Rerank -> Local LLM Generation."""
    
    # Step 1: Retrieve and rerank
    reranked = hybrid_search_with_rerank(query, top_n=top_n)
    
    # Step 2: Format retrieved chunks as context
    context_parts = []
    for i, (score, doc) in enumerate(reranked):
        meta = doc.metadata
        page = meta.get('page_number', '?')
        category = meta.get('category', 'Unknown')
        context_parts.append(
            f"--- Excerpt {i+1} (Page {page}, Category: {category}, Relevance: {score:.3f}) ---\n{doc.page_content}"
        )
    context_str = "\n\n".join(context_parts)
    
    # Step 3: Build messages
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context_str}\n\nQuestion:\n{query}\n\nAnswer:"},
    ]
    
    # Step 4: Configure and run generation
    # We create a specific GenerationConfig to avoid deprecation warnings
    gen_config = GenerationConfig(
        temperature=0.3,
        do_sample=True,
        pad_token_id=llm_pipeline.tokenizer.eos_token_id,
    )
    
    prompt = llm_pipeline.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    output = llm_pipeline(
        prompt, 
        generation_config=gen_config,
        return_full_text=False
    )
    return output[0]['generated_text'].strip()

print('RAG pipeline with local LLM generation is ready!')


Loading local LLM (TinyLlama-1.1B-Chat)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Local LLM ready!
RAG pipeline with local LLM generation is ready!


### Step 9: Ask the Grandmaster

In [15]:
question = "How does a pawn move?"

print(f"Question: {question}\n")
print("=" * 60)
answer = generate_answer(question)
print("ANSWER:\n")
print(answer)


Question: How does a pawn move?

ANSWER:

In chess, a pawn move is a move that a pawn makes on a chessboard. There are two types of pawn moves: forward and backward.

Forward Pawn Move:
- A pawn moves forward one square on the same file, or two squares on the same file if it is a bishop pawn.
- For example, if a white pawn moves to the square e2, it moves one square forward.
- The pawn can move diagonally, but it cannot move diagonally twice in one move.

Backward Pawn Move:
- A pawn moves backward one square on the same file, or two squares on the same file if it is a bishop pawn.
- For example, if a white pawn moves to the square g1, it moves one square backward.
- The pawn can move diagonally, but it cannot move diagonally twice in one move.

When a pawn moves, it captures any piece that is in the way. If a pawn captures a piece that is already captured, it captures that piece. If a pawn captures a piece that is not yet captured, it captures the piece after it has been released by t

In [16]:
# Try another query
question2 = "How does a rook move?"

print(f"Question: {question2}\n")
print("=" * 60)
answer2 = generate_answer(question2)
print("ANSWER:\n")
print(answer2)


Question: How does a rook move?

ANSWER:

A rook move is a move of the king and either rook of the same color along the player's first rank, counted as a single move of the king and executed as follows: the king is transferred from its original square two squares towards the rook on its original square, then that rook is transferred to the square the king has just crossed. 

The context of the question is a Grandmaster Strategist who is asked to answer a user's question about a specific rule in chess. The context is a section of the rules that discusses the movement of the king and rook. The given excerpt is from the section that discusses the effect of promoting a pawn. The section also includes the rules for exchanging pawns and promoting a pawn. The given excerpt is relevant to the question because it explains how a rook move works.
